# 01 제1유형

다음의 데이터는 IBM 직원들의 직무 정보와 퇴사 여부에 대한 데이터이다.

**데이터 URL**
```
https://raw.githubusercontent.com/YoungjinBD/dataset/main/HR-Employee-Attrition.csv
```

| 컬럼 | 설명 |
|---|---|
| Attrition | 퇴사 여부 (`Yes`: 퇴사, `No`: 퇴사하지 않음) |
| 기타 | 직무/인구통계/근무 관련 변수 |

In [4]:
import pandas as pd
import numpy as np

# 데이터 로드
data_url = "https://raw.githubusercontent.com/YoungjinBD/dataset/main/HR-Employee-Attrition.csv"
hr = pd.read_csv(data_url)
hr.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 문제 ① Attrition 수치화 및 범주별 건수

해당 데이터에서 `Attrition`은 종속변수이다.  
Attrition의 값을 **수치형**으로 변환해 새로운 컬럼으로 추가하고,  
범주별 **레코드 수**를 구하시오.

- `Yes` → `1` (퇴사)
- `No` → `0` (퇴사하지 않음)

> 출력: Attrition(0/1) 범주별 건수

In [5]:
hr['Attrition_num'] = hr['Attrition'].map({'Yes': 1, 'No': 0})
print(hr['Attrition_num'].value_counts())

Attrition_num
0    1233
1     237
Name: count, dtype: int64


## 문제 ② 자료형별 컬럼 수 및 단일값 범주형 제거

데이터셋에서 **자료형(dtype)별 컬럼 수**를 계산하고,  
범주형 변수 중 **유일한 값이 1개뿐인 변수**를 찾아 데이터에서 제거하시오.

> 출력: dtype별 컬럼 수, 제거 대상 컬럼명, 제거 후 데이터 shape

In [6]:
# 1) 자료형(dtype)별 컬럼 수
print(hr.dtypes.value_counts())

# 2) 범주형(object) 변수 중 유일값이 1개인 변수 찾기
cat = hr.select_dtypes(include='object')
drop_cols = [c for c in cat.columns if cat[c].nunique() == 1]
print('제거 대상:', drop_cols)

# 3) 제거
hr2 = hr.drop(columns=drop_cols)
print('제거 후 shape:', hr2.shape)



int64     27
object     9
Name: count, dtype: int64
제거 대상: ['Over18']
제거 후 shape: (1470, 35)


## 문제 ③ 수치형 상관분석 및 고상관 변수 제거

원본 데이터에서 **수치형 변수만** 추출한 데이터프레임을 만들고,  
각 변수 간 **피어슨 상관계수**를 구하시오.  
상관계수가 **0.9 이상**인 두 변수를 찾아 그 중 **한 변수를 제거**하시오.

> 출력: 상관계수 0.9 이상인 변수 쌍, 제거한 컬럼명, 제거 후 shape

In [7]:
# 1) 수치형 변수만 추출
num = hr.select_dtypes(include='number')

# 2) 피어슨 상관계수
corr = num.corr(method='pearson')

# 3) 상관계수 0.9 이상인 변수 쌍 찾기 (상삼각행렬만 확인, 자기 자신 제외)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = [(r, c, upper.loc[r, c])
         for r in upper.index for c in upper.columns
         if abs(upper.loc[r, c]) >= 0.9]
print('상관 0.9 이상 쌍:', pairs)

# 4) 각 쌍에서 뒤쪽 변수 하나씩 제거
drop_cols = list({c for _, c, _ in pairs})
print('제거한 컬럼:', drop_cols)

num2 = num.drop(columns=drop_cols)
print('제거 후 shape:', num2.shape)



상관 0.9 이상 쌍: [('JobLevel', 'MonthlyIncome', np.float64(0.9502999134798473))]
제거한 컬럼: ['MonthlyIncome']
제거 후 shape: (1470, 26)
